# CodeAct: Code Agents

The paper [Executable Code Actions Elicit Better LLM Agents](https://arxiv.org/abs/2402.01030) shows that planning, reasoning, and solving tasks through coding is actually way better than writing text, or spitting out JSON to call tools.

> "We propose to use executable Python code to consolidate LLM agents’ actions into a unified space... LLM agents can leverage existing software packages and iteratively adjust their actions through code execution feedback."

![](../assets/CodeAct_1.png)

![](../assets/CodeAct_2.png)

![](../assets/CodeAct_3.png)

## Setting up DSPy

### Load environment variables

In [2]:
import os

try:
    # In Colab? read from userdata (secrets)
    from google.colab import userdata
    ON_COLAB = True
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
except ImportError:
    from pathlib import Path

    if not Path('.env').exists():
        print("No .env file found. Please create one with OPENROUTER_API_KEY.")

    from dotenv import load_dotenv
    load_dotenv(override=True)

print(len(os.environ["OPENROUTER_API_KEY"]))

73


### Install Library

In [3]:
# ! uv add dspy
# ! pip install dspy

### Connecting to a language model

DSPy connects to language models with the `dspy.LM` class. To set up a language model, we provide a `"provider/model"` format string and an API key:


In [4]:
import dspy

# Pass the key explicitly...
lm = dspy.LM(
    "openrouter/openai/gpt-5.4-mini",
    api_key=os.environ["OPENROUTER_API_KEY"]
)

dspy.configure(lm=lm)

## Give the program ability to think in code with `dspy.CodeAct`

[**CodeAct**](https://dspy.ai/api/modules/CodeAct/) is a module that utilizes the _Code Interpreter_ and predefined tools to solve the problem.


### Module Composition

`CodeAct` is a module that we shall see in the next lesson.

It extends both `ReAct` and `ProgramOfThought`. As show in the figure below:


![](../assets/CodeAct.png)

### How It Works

`CodeAct` operates in an iterative manner:

1. Takes input parameters and available tools
2. Generates Python code snippets that use these tools
3. Executes the code using a host Python interpreter
4. Collects the output and determines if the task is complete
5. Answer the original question based on the collected information

![](../assets/CodeAct_4.png)

### Code Execution

The ready-made `PythonInterpreter` from DSPy executes code in a WASM sandbox. We have implemented our own unsandboxed `PythonInterpreterLocal`, compatible with the `CodeAct` module. Let's import it:

In [5]:
from ext.python_interpreter import PythonInterpreterLocal

#### Local interpreter test run

Let's type in some code and check that it is working:

In [6]:
interpreter = PythonInterpreterLocal()

code = """
def add(a, b):
    return a + b

result = add(3, 4)
print("Sum:", result)
"""

output = interpreter.execute(code)
print(output)

Sum: 7


#### Passing in Function Definitions

How about defining a function:

In [7]:
def favorite_color(name: str) -> str:
    """Return the favorite color of a given person."""
    m = {
        "Alice": "red",
        "Bob": "blue"
    }
    return m.get(name, "unknown")

interpreter = PythonInterpreterLocal(tools={"favorite_color": favorite_color})

code = """
print(favorite_color("Alice"))
"""

output = interpreter.execute(code)
print(output)

red


### Passing in modules via `namespace`

`PythonInterpreterLocal` accepts a `namespace` list of modules, callables, dicts, or other injectables available during code execution.

Let's start with the stdlib [`datetime`](https://docs.python.org/3/library/datetime.html) module:

In [8]:
import datetime

interpreter = PythonInterpreterLocal(namespace=[datetime])

code = """
today = datetime.date.today()
print(today.strftime("%Y-%m-%d"))
"""

print(interpreter.execute(code))

2026-06-17


### Passing installed modules

We'll install [`python-dateutil`](https://pypi.org/project/python-dateutil/) library from PyPI, and test importing it into the interpreter, and using it: 

In [9]:
import dateutil

interpreter = PythonInterpreterLocal(namespace=[dateutil])

code = """
from dateutil.parser import parse
parsed = parse("June 17, 2026")
print(parsed.year)
"""

print(interpreter.execute(code))

2026


### CodeAct for date and time

Now we put `PythonInterpreterLocal` and [`CodeAct`](https://dspy.ai/api/modules/CodeAct/) together to answer increasingly tricky date/time questions.

- One `CodeAct("question -> answer", ...)` module handles every question below.
- Modules are injected into the interpreter **namespace** — the agent writes Python against them.
- We add libraries only when the stdlib gets awkward; the namespace grows as difficulty ramps up.


In [10]:
import datetime

from dspy.predict import CodeAct

date_interpreter = PythonInterpreterLocal(
    namespace=[datetime],
)

def execute_code(code: str) -> str:
    return date_interpreter.execute(code)

date_act = CodeAct(
    signature="question -> answer",
    tools=[execute_code],
    interpreter=date_interpreter,
)


In [11]:
import utils


def ask_date(question: str) -> dspy.Prediction:
    result = date_act(question=question)
    utils.print_pretty_codeact_trajectory(result, result.answer)
    return result


#### Level 1 — age in days

A straight `datetime.date` subtraction — no extra libraries needed beyond what we already injected.


In [14]:
from importlib import reload

reload(utils)

<module 'utils' from '/home/halgoz/work/Bootcamp/ai-pros/public/courses/Building_with_Agentic_AI/lessons/utils.py'>

In [15]:
result = ask_date("How old am I if I was born on January 1st, 2000?")

2026/06/17 09:57:14 WARNING dspy.predict.predict: Type mismatch for field 'trajectory': expected str based on given Signature, but the provided value is incompatible: {}.


📝 generated_code:

from datetime import date

birth = date(2000, 1, 1)
today = date.today()
age = today.year - birth.year - ((today.month, today.day) < (birth.month, birth.day))
print(age)

📤 code_output: "26"

✅ answer: 26

In [17]:
print(result.answer)

26


#### Level 2 — countdown to a future date

Same building blocks: parse the target date, subtract from today, and express the remaining time clearly.


In [12]:
ask_date("How much time is left until 2045, June, 1st?")


2026/06/17 09:23:28 WARNING dspy.predict.predict: Type mismatch for field 'trajectory': expected str based on given Signature, but the provided value is incompatible: {}.


Time left until 2045-06-01 (UTC): 6923 days, 17 hours, 36 minutes, 29 seconds


📤 code_output: "Time left until 2045-06-01 (UTC): 6923 days, 17 hours, 36 minutes, 29 seconds"

📝 generated_code: from datetime import datetime, timezone

target = datetime(2045, 6, 1, tzinfo=timezone.utc)
now = datetime.now(timezone.utc)
delta = target - now

days = delta.days
seconds = delta.seconds
hours, rem = divmod(seconds, 3600)
minutes, secs = divmod(rem, 60)

print(f"Time left until 2045-06-01 (UTC): {days} days, {hours} hours, {minutes} minutes, {secs} seconds")

Prediction(
    trajectory={'generated_code_0': 'from datetime import datetime, timezone\n\ntarget = datetime(2045, 6, 1, tzinfo=timezone.utc)\nnow = datetime.now(timezone.utc)\ndelta = target - now\n\ndays = delta.days\nseconds = delta.seconds\nhours, rem = divmod(seconds, 3600)\nminutes, secs = divmod(rem, 60)\n\nprint(f"Time left until 2045-06-01 (UTC): {days} days, {hours} hours, {minutes} minutes, {secs} seconds")', 'code_output_0': '"Time left until 2045-06-01 (UTC): 6923 days, 17 hours, 36 minutes, 29 seconds"'},
    reasoning='The provided code computes the time remaining until 2045-06-01 in UTC and prints the result directly. We should return that computed output as the answer.',
    answer='Time left until 2045-06-01 (UTC): 6923 days, 17 hours, 36 minutes, 29 seconds'
)

#### Level 3 — days until next Friday


In [13]:
ask_date("How many days until next Friday?")


2026/06/17 09:23:41 WARNING dspy.predict.predict: Type mismatch for field 'trajectory': expected str based on given Signature, but the provided value is incompatible: {}.


2


📤 code_output: "2"

📝 generated_code: from datetime import date

today = date.today()
# Monday=0 ... Sunday=6
days_until_friday = (4 - today.weekday()) % 7
print(days_until_friday)

Prediction(
    trajectory={'generated_code_0': 'from datetime import date\n\ntoday = date.today()\n# Monday=0 ... Sunday=6\ndays_until_friday = (4 - today.weekday()) % 7\nprint(days_until_friday)', 'code_output_0': '"2"'},
    reasoning='The provided code computes the number of days until Friday by taking today\'s weekday and calculating the modular difference to Friday. The code output is `"2"`, so the answer is 2 days.',
    answer='2'
)

#### Level 4 — days since last Friday

The mirror of the previous question. If today is Friday, different conventions give `0` or `7` — let the agent state which one it uses.


In [14]:
ask_date("How many days since last Friday?")


2026/06/17 09:23:53 WARNING dspy.predict.predict: Type mismatch for field 'trajectory': expected str based on given Signature, but the provided value is incompatible: {}.


5


📤 code_output: "5"

📝 generated_code: from datetime import datetime

today = datetime.now().date()
# Monday=0, Tuesday=1, ..., Sunday=6; Friday=4
days_since_friday = (today.weekday() - 4) % 7
print(days_since_friday)

Prediction(
    trajectory={'generated_code_0': 'from datetime import datetime\n\ntoday = datetime.now().date()\n# Monday=0, Tuesday=1, ..., Sunday=6; Friday=4\ndays_since_friday = (today.weekday() - 4) % 7\nprint(days_since_friday)', 'code_output_0': '"5"'},
    reasoning='The provided code computes the number of days since Friday using the current date. The code output is `5`, so the answer is 5 days.',
    answer='5'
)

So far, `datetime` was enough. **Time zones** need the stdlib [`zoneinfo`](https://docs.python.org/3/library/zoneinfo.html) module — let's add it to the namespace and rebuild `CodeAct`.


In [ ]:
import zoneinfo

date_interpreter = PythonInterpreterLocal(
    namespace=[datetime, zoneinfo],
)
date_act = CodeAct(
    signature="question -> answer",
    tools=[execute_code],
    interpreter=date_interpreter,
)


#### Level 5 — Cairo → London flight

Departure is **1:00 PM** on 2044-11-19 in Cairo (`Africa/Cairo`). The flight lasts four hours. We want the wall-clock time at arrival in both London (`Europe/London`) and Cairo.


In [16]:
ask_date(
    "If departure is on 2044, 11, 19 at 1:00 PM Cairo time and arrival "
    "in London is 4 hours later, what time is it at arrival in London and in Cairo?"
)


2026/06/17 09:24:54 WARNING dspy.predict.predict: Type mismatch for field 'trajectory': expected str based on given Signature, but the provided value is incompatible: {}.


Arrival in London: 2044-11-19 03:00 PM GMT
Arrival in Cairo: 2044-11-19 05:00 PM EET


📤 code_output: "Arrival in Cairo: 2044-11-19 05:00 PM EET\nArrival in London: 2044-11-19 03:00 PM GMT"

📝 generated_code: from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

# Departure: 2044-11-19 1:00 PM Cairo time
cairo_tz = ZoneInfo("Africa/Cairo")
london_tz = ZoneInfo("Europe/London")

departure_cairo = datetime(2044, 11, 19, 13, 0, tzinfo=cairo_tz)
arrival_cairo = departure_cairo + timedelta(hours=4)
arrival_london = arrival_cairo.astimezone(london_tz)

print("Arrival in Cairo:", arrival_cairo.strftime("%Y-%m-%d %I:%M %p %Z"))
print("Arrival in London:", arrival_london.strftime("%Y-%m-%d %I:%M %p %Z"))

Prediction(
    trajectory={'generated_code_0': 'from datetime import datetime, timedelta\nfrom zoneinfo import ZoneInfo\n\n# Departure: 2044-11-19 1:00 PM Cairo time\ncairo_tz = ZoneInfo("Africa/Cairo")\nlondon_tz = ZoneInfo("Europe/London")\n\ndeparture_cairo = datetime(2044, 11, 19, 13, 0, tzinfo=cairo_tz)\narrival_cairo = departure_cairo + timedelta(hours=4)\narrival_london = arrival_cairo.astimezone(london_tz)\n\nprint("Arrival in Cairo:", arrival_cairo.strftime("%Y-%m-%d %I:%M %p %Z"))\nprint("Arrival in London:", arrival_london.strftime("%Y-%m-%d %I:%M %p %Z"))', 'code_output_0': '"Arrival in Cairo: 2044-11-19 05:00 PM EET\\nArrival in London: 2044-11-19 03:00 PM GMT"'},
    reasoning='The departure is 1:00 PM in Cairo. Adding 4 hours gives 5:00 PM in Cairo. Converting that same moment to London time gives 3:00 PM in London.',
    answer='Arrival in London: 2044-11-19 03:00 PM GMT\nArrival in Cairo: 2044-11-19 05:00 PM EET'
)

#### Astronomical times — when stdlib is not enough

[`astral`](https://astral.readthedocs.io/) computes sunrise, sunset, and dawn for a latitude/longitude.

For **Islamic night divisions** in Riyadh:

- **Night** runs from sunset (maghrib) to dawn (fajr).
- **Islamic midnight** (*nisful layl*) is the midpoint of that interval — not clock `00:00`.
- **First third of the night** = sunset + (dawn − sunset) / 3.
- **Last third of the night** = sunset + 2 × (dawn − sunset) / 3.


In [ ]:
import astral

date_interpreter = PythonInterpreterLocal(
    namespace=[datetime, zoneinfo, astral],
)
date_act = CodeAct(
    signature="question -> answer",
    tools=[execute_code],
    interpreter=date_interpreter,
    max_iters=8,  # more steps for sunrise/sunset + night-thirds math
)


#### Level 6 — Riyadh sunrise, sunset, and night thirds

Ask for all five times **today** in Riyadh (`Asia/Riyadh`). The answer depends on when you run the notebook.


In [18]:
result = ask_date(
    "For today in Riyadh time, when are sunrise, sunset, Islamic midnight "
    "(middle of the night), the first third of the night, and the last third of the night?"
)
print(result.answer)

2026/06/17 09:26:27 WARNING dspy.predict.predict: Type mismatch for field 'trajectory': expected str based on given Signature, but the provided value is incompatible: {}.


For today in Riyadh time:

- Sunrise: 2026-06-17 05:04:23 +03
- Sunset: 2026-06-17 18:44:05 +03
- Islamic midnight (middle of the night): 2026-06-17 23:54:19 +03
- First third of the night: 2026-06-17 22:10:54 +03
- Last third of the night: 2026-06-18 01:37:43 +03


📤 code_output: "Today (Riyadh): 2026-06-17\nSunrise: 2026-06-17 05:04:23 +03\nSunset: 2026-06-17 18:44:05 
+03\nIslamic midnight (middle of the night): 2026-06-17 23:54:19 +03\nFirst third of the night: 2026-06-17 22:10:54
+03\nLast third of the night: 2026-06-18 01:37:43 +03"

📝 generated_code: from datetime import datetime
from zoneinfo import ZoneInfo

try:
    from astral import LocationInfo
    from astral.sun import sun
    from astral.moon import moonrise, moonset
except ImportError:
    print("astral library is required")
    raise

# Riyadh coordinates
city = LocationInfo("Riyadh", "Saudi Arabia", "Asia/Riyadh", 24.7136, 46.6753)
tz = ZoneInfo("Asia/Riyadh")

today = datetime.now(tz).date()

s = sun(city.observer, date=today, tzinfo=tz)
sunrise = s["sunrise"]
sunset = s["sunset"]

# Night calculations:
# Night starts at sunset and ends at next sunrise.
# Islamic midnight = halfway between sunset and next sunrise.
# First third of night = sunset + 1/3 of night duration
# Last third of night = sunset + 2/3 of night duration
from datetime import timedelta

# sunrise for next day
from datetime import date as date_cls
next_day = today + timedelta(days=1)
next_sun = sun(city.observer, date=next_day, tzinfo=tz)
next_sunrise = next_sun["sunrise"]

night_duration = next_sunrise - sunset
middle_of_night = sunset + night_duration / 2
first_third = sunset + night_duration / 3
last_third = sunset + night_duration * 2 / 3

print(f"Today (Riyadh): {today}")
print(f"Sunrise: {sunrise.strftime('%Y-%m-%d %H:%M:%S %Z')}")
print(f"Sunset: {sunset.strftime('%Y-%m-%d %H:%M:%S %Z')}")
print(f"Islamic midnight (middle of the night): {middle_of_night.strftime('%Y-%m-%d %H:%M:%S %Z')}")
print(f"First third of the night: {first_third.strftime('%Y-%m-%d %H:%M:%S %Z')}")
print(f"Last third of the night: {last_third.strftime('%Y-%m-%d %H:%M:%S %Z')}")

For today in Riyadh time:

- Sunrise: 2026-06-17 05:04:23 +03
- Sunset: 2026-06-17 18:44:05 +03
- Islamic midnight (middle of the night): 2026-06-17 23:54:19 +03
- First third of the night: 2026-06-17 22:10:54 +03
- Last third of the night: 2026-06-18 01:37:43 +03
